# 04 — DiT denoiser & c_spec conditioning

Verify that the DiT denoiser is sensitive to `c_spec` conditioning and demonstrate CFG dropout.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import matplotlib.pyplot as plt
from ald_sc.dit import MinimalDiT

torch.manual_seed(3407)

## 1. DiT forward pass

In [ ]:
model = MinimalDiT(
    latent_channels=4,
    latent_size=32,
    patch_size=2,
    dim=128,
    depth=4,
    num_heads=4,
    text_dim=0,
    spec_dim=24,
    cfg_dropout=0.1,
)
z = torch.randn(2, 4, 32, 32)
t = torch.tensor([100, 500])
c_spec = torch.randn(2, 24)

out = model(z, t, c_spec=c_spec)
print(f"Input:  {z.shape}")
print(f"Output: {out.shape}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

## 2. Conditioning sensitivity: swap c_spec between images

In [ ]:
model.eval()
z = torch.randn(4, 4, 32, 32)
t = torch.tensor([100, 200, 300, 400])
c_a = torch.randn(4, 24)
c_b = torch.randn(4, 24)

with torch.no_grad():
    out_a = model(z, t, c_spec=c_a)
    out_b = model(z, t, c_spec=c_b)
    diff = (out_a - out_b).abs().mean()

print(f"Mean |output_a - output_b| when only c_spec changes: {diff:.6f}")
assert diff > 1e-4, "c_spec must affect output"

## 3. CFG dropout: effect of dropping c_spec

In [ ]:
model.train()
z = torch.randn(8, 4, 32, 32)
t = torch.randint(0, 1000, (8,))
c_spec = torch.randn(8, 24)

dropout_probs = [0.0, 0.25, 0.5, 0.75, 1.0]
diffs = []
for p in dropout_probs:
    m = MinimalDiT(latent_channels=4, latent_size=32, patch_size=2, dim=64, depth=2,
                   num_heads=4, spec_dim=24, cfg_dropout=p)
    m.train()
    with torch.no_grad():
        out_cond = m(z, t, c_spec=c_spec)
        out_uncond = m(z, t, c_spec=torch.zeros_like(c_spec))
    diff = (out_cond - out_uncond).abs().mean()
    diffs.append(diff.item())

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(len(dropout_probs)), diffs)
ax.set_xticks(range(len(dropout_probs)))
ax.set_xticklabels([str(p) for p in dropout_probs])
ax.set_xlabel("cfg_dropout prob")
ax.set_ylabel("|cond - uncond| output")
ax.set_title("CFG dropout effect")
plt.tight_layout()
plt.savefig("../results/04_cfg_dropout.png", dpi=150)
plt.show()

## 4. Velocity prediction at different timesteps

In [ ]:
model.eval()
z = torch.randn(1, 4, 32, 32)
c_spec = torch.randn(1, 24)
timesteps = [0, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(timesteps), figsize=(18, 3))
for i, t_val in enumerate(timesteps):
    with torch.no_grad():
        v = model(z, torch.tensor([t_val]), c_spec=c_spec)
    axes[i].imshow(v[0, 0].numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
    axes[i].set_title(f"t={t_val}")
    axes[i].axis("off")
plt.suptitle("Predicted velocity v at different timesteps")
plt.tight_layout()
plt.savefig("../results/04_velocity_timesteps.png", dpi=150)
plt.show()